### 1.  Import libraries

In [1]:
import sqlite3
import pandas as pd
import os
from datetime import datetime

### 2. Load transformed data

In [2]:

df = pd.read_csv('../data/transformed_weather.csv')

print(f"Loaded {len(df)} records to insert into database")
print(f"Columns: {list(df.columns)}")
print("\nFirst 2 records:")
df.head(2)

Loaded 6 records to insert into database
Columns: ['timestamp', 'city', 'country', 'temperature_c', 'feels_like_c', 'temp_min_c', 'temp_max_c', 'humidity_pct', 'pressure_hpa', 'weather_main', 'weather_description', 'wind_speed_ms', 'wind_deg', 'clouds_pct', 'visibility_m', 'temp_fahrenheit', 'temp_range_c', 'heat_index_c']

First 2 records:


,timestamp,city,country,temperature_c,feels_like_c,temp_min_c,temp_max_c,humidity_pct,pressure_hpa,weather_main,weather_description,wind_speed_ms,wind_deg,clouds_pct,visibility_m,temp_fahrenheit,temp_range_c,heat_index_c
0,2026-04-19 17:18:24,London,GB,15.87,14.45,14.44,16.83,36,1024,Clouds,overcast clouds,2.68,50,88,10000,60.6,2.4,15.4
1,2026-04-19 17:18:24,New York,US,11.69,10.79,10.32,12.34,72,1011,Mist,mist,10.29,330,100,10000,53.0,2.0,12.1


### 3. Create database and table

In [3]:
# Connect to database (creates file if it doesn't exist)
conn = sqlite3.connect('../data/weather_data.db')
cursor = conn.cursor()

# Create table with appropriate schema
cursor.execute('''
    CREATE TABLE IF NOT EXISTS weather_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        city TEXT,
        country TEXT,
        temperature_c REAL,
        feels_like_c REAL,
        temp_min_c REAL,
        temp_max_c REAL,
        temp_fahrenheit REAL,
        temp_range_c REAL,
        heat_index_c REAL,
        humidity_pct INTEGER,
        pressure_hpa INTEGER,
        weather_main TEXT,
        weather_description TEXT,
        wind_speed_ms REAL,
        wind_deg INTEGER,
        clouds_pct INTEGER,
        visibility_m INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
''')

print("Database and table ready")
print(f"Database location: data/weather_data.db")
print(f"Table name: weather_log")

Database and table ready
Database location: data/weather_data.db
Table name: weather_log


### 4. Check if records already exist (avoid duplicates)

In [4]:
# Check existing records
cursor.execute("SELECT COUNT(*) FROM weather_log")
existing_count = cursor.fetchone()[0]
print(f"Existing records in database: {existing_count}")

# Show latest records if any
if existing_count > 0:
    cursor.execute("SELECT city, temperature_c, timestamp FROM weather_log ORDER BY id DESC LIMIT 3")
    print("\nLatest records in database:")
    for row in cursor.fetchall():
        print(f"  {row[0]}: {row[1]}C at {row[2]}")

Existing records in database: 0


### 5. Insert data into database

In [5]:
# Insert each row
records_inserted = 0
records_skipped = 0

print("Inserting records...")
print("-" * 40)

for _, row in df.iterrows():
    # Check if record already exists (same city and timestamp within 1 hour)
    cursor.execute('''
        SELECT COUNT(*) FROM weather_log 
        WHERE city = ? AND datetime(timestamp) >= datetime(?)
    ''', (row['city'], row['timestamp']))
    
    if cursor.fetchone()[0] == 0:
        cursor.execute('''
            INSERT INTO weather_log (
                timestamp, city, country, temperature_c, feels_like_c,
                temp_min_c, temp_max_c, temp_fahrenheit, temp_range_c, heat_index_c,
                humidity_pct, pressure_hpa, weather_main, weather_description,
                wind_speed_ms, wind_deg, clouds_pct, visibility_m
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            row['timestamp'], row['city'], row['country'],
            row['temperature_c'], row['feels_like_c'],
            row['temp_min_c'], row['temp_max_c'],
            row['temp_fahrenheit'], row['temp_range_c'], row['heat_index_c'],
            row['humidity_pct'], row['pressure_hpa'],
            row['weather_main'], row['weather_description'],
            row['wind_speed_ms'], row['wind_deg'],
            row['clouds_pct'], row['visibility_m']
        ))
        records_inserted += 1
        print(f"INSERTED: {row['city']:15} | {row['temperature_c']:5.1f}C | {row['timestamp']}")
    else:
        records_skipped += 1
        print(f"SKIPPED:  {row['city']:15} | Already exists")

conn.commit()
print("-" * 40)
print(f"Insert complete!")
print(f"  Records inserted: {records_inserted}")
print(f"  Records skipped (duplicates): {records_skipped}")

Inserting records...
----------------------------------------
INSERTED: London          |  15.9C | 2026-04-19 17:18:24
INSERTED: New York        |  11.7C | 2026-04-19 17:18:24
INSERTED: Tokyo           |  17.5C | 2026-04-19 17:18:24
INSERTED: Sydney          |  12.4C | 2026-04-19 17:18:24
INSERTED: Cape Town       |  14.2C | 2026-04-19 17:18:24
INSERTED: Mumbai          |  30.0C | 2026-04-19 17:18:24
----------------------------------------
Insert complete!
  Records inserted: 6
  Records skipped (duplicates): 0


### 6. Verify insertion

In [6]:
# Query to verify data was inserted
query = "SELECT COUNT(*) as total FROM weather_log"
total_count = pd.read_sql_query(query, conn).iloc[0]['total']

print(f"Total records in database: {total_count}")

# Show the most recent records
query = "SELECT * FROM weather_log ORDER BY id DESC LIMIT 5"
verify_df = pd.read_sql_query(query, conn)

print("\nMost recent records in database:")
verify_df[['id', 'city', 'temperature_c', 'humidity_pct', 'weather_description', 'timestamp']]

Total records in database: 6

Most recent records in database:


,id,city,temperature_c,humidity_pct,weather_description,timestamp
0,6,Mumbai,29.99,66,haze,2026-04-19 17:18:24
1,5,Cape Town,14.20,74,overcast clouds,2026-04-19 17:18:24
2,4,Sydney,12.42,78,broken clouds,2026-04-19 17:18:24
3,3,Tokyo,17.46,77,broken clouds,2026-04-19 17:18:24
4,2,New York,11.69,72,mist,2026-04-19 17:18:24


### 7.  Display database statistics

In [7]:
# Database statistics
stats_query = """
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT city) as unique_cities,
        AVG(temperature_c) as avg_temp_c,
        MIN(temperature_c) as min_temp_c,
        MAX(temperature_c) as max_temp_c,
        AVG(humidity_pct) as avg_humidity,
        datetime(MIN(created_at)) as first_record,
        datetime(MAX(created_at)) as last_record
    FROM weather_log
"""

stats_df = pd.read_sql_query(stats_query, conn)
print("Database Statistics:")
print("=" * 50)
for col in stats_df.columns:
    value = stats_df.iloc[0][col]
    if isinstance(value, float):
        print(f"{col:20}: {value:.2f}")
    else:
        print(f"{col:20}: {value}")

Database Statistics:
total_records       : 6
unique_cities       : 6
avg_temp_c          : 16.94
min_temp_c          : 11.69
max_temp_c          : 29.99
avg_humidity        : 67.17
first_record        : 2026-04-19 15:44:20
last_record         : 2026-04-19 15:44:20


### 8. Query by city

In [10]:
# Query weather data for a specific city
city_to_query = "London"

city_query = f"SELECT * FROM weather_log WHERE city = '{city_to_query}' ORDER BY id DESC LIMIT 5"
city_df = pd.read_sql_query(city_query, conn)

print(f"Weather records for {city_to_query}:")
print("-" * 50)
if len(city_df) > 0:
    print(city_df[['timestamp', 'temperature_c', 'humidity_pct', 'weather_description']].to_string(index=False))
else:
    print(f"No records found for {city_to_query}")
    print("Make sure data was inserted successfully in previous cells")

Weather records for London:
--------------------------------------------------
          timestamp  temperature_c  humidity_pct weather_description
2026-04-19 17:18:24          15.87            36     overcast clouds


### 9.  Close database connection

In [11]:
# Close the database connection
conn.close()
print("Database connection closed")

Database connection closed


### 11. Load stage summary

In [13]:
print("Load Stage Complete!")
print("=" * 50)
print(f"Records processed: {len(df)}")
print(f"Records inserted: {records_inserted}")
print(f"Records skipped: {records_skipped}")
print("\nDatabase file created:")
print(f"  - data/weather_data.db")

Load Stage Complete!
Records processed: 6
Records inserted: 6
Records skipped: 0

Database file created:
  - data/weather_data.db
